# Read-only Drive Satellite Hydrator v2

This notebook is a manually triggered **pilot**. It does **not** classify safety loops, publish contractor releases, or push GitHub. Run each cell deliberately after reviewing paths and content.


In [ ]:
from pathlib import Path
import base64, hashlib, json, re

# OPTIONAL: In Google Colab, uncomment the following two lines after user authorization.
# from google.colab import drive
# drive.mount('/content/drive')

PILOT_DIR = Path('/content/drive/MyDrive/YOUR_VERIFIED_WORKTREE')
print('Set PILOT_DIR to the verified mounted folder; exists:', PILOT_DIR.exists())


In [ ]:
def verify_raw_file(path, expected_sha256):
    if not isinstance(expected_sha256,str) or not re.fullmatch(r'[0-9a-fA-F]{64}',expected_sha256):
        return 'NOT_VERIFIED: missing/invalid SHA256'
    path=Path(path)
    if not path.is_file():
        return 'NOT_VERIFIED: original raw bytes unavailable'
    actual=hashlib.sha256(path.read_bytes()).hexdigest()
    return 'PASS' if actual.lower()==expected_sha256.lower() else 'FAIL: digest mismatch'



In [ ]:
def safe_relative_path(s):
    from pathlib import PurePosixPath
    if not isinstance(s,str) or '\\' in s or ':' in s or s.startswith('/') or chr(0) in s:
        raise ValueError('Invalid relative path')
    p=PurePosixPath(s)
    if any(x in ('.','..','') for x in s.split('/')) or not p.parts:
        raise ValueError('Path traversal or ambiguous relative path')
    return p

def hydrate_bundle(bundle_path, output_dir, execute=False):
    bundle=json.loads(Path(bundle_path).read_text(encoding='utf-8'))
    rows=bundle.get('files',[])
    if not isinstance(rows,list) or len(rows)>100:
        raise ValueError('Bundle file count exceeds limit')
    total=0; checked=[]; seen=set()
    for row in rows:
        rel=safe_relative_path(row['path'])
        if str(rel) in seen: raise ValueError('Duplicate path')
        seen.add(str(rel))
        if row.get('encoding')!='base64': raise ValueError('Unsupported encoding')
        data=base64.b64decode(row['data'],validate=True)
        if len(data)>2_000_000: raise ValueError('File too large')
        total+=len(data)
        if total>10_000_000: raise ValueError('Total payload too large')
        digest=hashlib.sha256(data).hexdigest()
        if digest!=row.get('sha256'): raise ValueError('Digest mismatch')
        checked.append((rel,data))
    target=Path(output_dir)
    if execute:
        if target.exists(): raise ValueError('Target exists; refuse overwrite')
        target.mkdir(parents=True,exist_ok=False)
        for rel,data in checked:
            dest=target.joinpath(*rel.parts)
            dest.parent.mkdir(parents=True,exist_ok=True)
            if dest.is_symlink() or dest.exists(): raise ValueError('Existing/symlink target refused')
            dest.write_bytes(data)
    return {'files_checked':len(checked),'total_bytes':total,'wrote_files':execute}

print(hydrate_bundle(PILOT_DIR/'synthetic_bundle.json',PILOT_DIR/'hydrated_satellite_v2',execute=False))
# To write, audit bundle contents first, then deliberately use execute=True.


## GitHub sync gate (disabled)

No automatic git pull/push is implemented. Compare with the existing federation bridge, get an exact source SHA and independent review, then create a separate owner-approved commit/PR. Never put access tokens in this notebook.
